In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Crear un plugin para el transpilador

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>Versiones de los paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o superiores.

```
qiskit[all]~=2.3.0
```
</details>
Crear un [plugin para el transpilador](transpiler-plugins) es una excelente manera de compartir tu código de transpilación con la comunidad de Qiskit, permitiendo que otros usuarios se beneficien de la funcionalidad que has desarrollado. ¡Gracias por tu interés en contribuir a la comunidad de Qiskit!

Antes de crear un plugin para el transpilador, debes decidir qué tipo de plugin es el adecuado para tu situación. Hay tres tipos de plugins:
- [**Plugin de etapa del transpilador**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins). Elige este si estás definiendo un pass manager que puede sustituir a una de las [6 etapas](transpiler-stages) de un pass manager de etapas preestablecido.
- [**Plugin de síntesis unitaria**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin). Elige este si tu código de transpilación recibe como entrada una matriz unitaria (representada como un array de Numpy) y devuelve la descripción de un circuito cuántico que implementa dicha unitaria.
- [**Plugin de síntesis de alto nivel**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin). Elige este si tu código de transpilación recibe como entrada un "objeto de alto nivel", como un operador Clifford o una función lineal, y devuelve la descripción de un circuito cuántico que implementa dicho objeto. Los objetos de alto nivel están representados por subclases de la clase [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

Una vez que hayas determinado qué tipo de plugin crear, sigue estos pasos:

1. Crea una subclase de la clase abstracta de plugin correspondiente:
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) para un plugin de etapa del transpilador,
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) para un plugin de síntesis unitaria, y
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) para un plugin de síntesis de alto nivel.
2. Expón la clase como un [entry point de setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) en los metadatos del paquete, generalmente editando el archivo `pyproject.toml`, `setup.cfg` o `setup.py` de tu paquete de Python.

No hay límite en el número de plugins que puede definir un solo paquete, pero cada plugin debe tener un nombre único. El propio SDK de Qiskit incluye varios plugins cuyos nombres están reservados:

- Plugins de etapa del transpilador: consulta [esta tabla](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- Plugins de síntesis unitaria: `default`, `aqc`, `sk`
- Plugins de síntesis de alto nivel:

| Clase de operación | Nombre de operación | Nombres reservados |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

En las secciones siguientes se muestran ejemplos de estos pasos para los distintos tipos de plugins. En estos ejemplos, asumimos que estamos creando un paquete de Python llamado `my_qiskit_plugin`. Para información sobre cómo crear paquetes de Python, puedes consultar [este tutorial](https://packaging.python.org/en/latest/tutorials/packaging-projects/) en el sitio web de Python.
## Ejemplo: Crear un plugin de etapa del transpilador
En este ejemplo, creamos un plugin de etapa del transpilador para la etapa `layout` (consulta [Etapas del transpilador](transpiler-stages) para una descripción de las 6 etapas del pipeline de transpilación integrado en Qiskit).
Nuestro plugin simplemente ejecuta [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) con un número de intentos que depende del nivel de optimización solicitado.

Primero, creamos una subclase de [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). Hay un método que debemos implementar, llamado [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). Este método recibe como entrada un [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) y devuelve el pass manager que estamos definiendo. El objeto PassManagerConfig almacena información sobre el backend de destino, como su mapa de acoplamiento y sus puertas base.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

Ahora, exponemos el plugin añadiendo un entry point en los metadatos de nuestro paquete de Python.
Aquí asumimos que la clase que definimos está expuesta en un módulo llamado `my_qiskit_plugin`, por ejemplo importándola en el archivo `__init__.py` del módulo `my_qiskit_plugin`.
Editamos el archivo `pyproject.toml`, `setup.cfg` o `setup.py` de nuestro paquete (según el tipo de archivo que hayas elegido para almacenar los metadatos de tu proyecto Python):

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>
Consulta la [tabla de etapas de plugins del transpilador](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table) para ver los entry points y los requisitos de cada etapa.

Para verificar que Qiskit detecta correctamente tu plugin, instala el paquete del plugin y sigue las instrucciones en [Plugins del transpilador](transpiler-plugins#list-available-transpiler-stage-plugins) para listar los plugins instalados, asegurándote de que tu plugin aparece en la lista:

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

Si nuestro plugin de ejemplo estuviera instalado, el nombre `my_layout` aparecería en esta lista.

Si quieres usar una etapa del transpilador integrada como punto de partida para tu plugin, puedes obtener el pass manager de esa etapa usando [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). El siguiente bloque de código muestra cómo obtener la etapa de optimización integrada para el nivel de optimización 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## Ejemplo: Crear un plugin de síntesis unitaria
En este ejemplo, crearemos un plugin de síntesis unitaria que simplemente usa el paso de transpilación integrado [UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) para sintetizar una puerta. Por supuesto, tu propio plugin hará algo más interesante que eso.

La clase [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) define la interfaz y el contrato para los plugins de síntesis unitaria. El método principal es
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
que recibe como entrada un array de Numpy con una matriz unitaria
y devuelve un [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit) que representa el circuito sintetizado a partir de esa matriz unitaria.
Además del método `run`, hay una serie de métodos de propiedad que deben definirse.
Consulta [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) para ver la documentación de todas las propiedades requeridas.

Vamos a crear nuestra subclase de UnitarySynthesisPlugin:

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

Si los datos de entrada disponibles para el método [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)
son insuficientes para tus necesidades, por favor [abre un issue](https://github.com/Qiskit/qiskit/issues/new/choose) explicando tus requisitos. Los cambios en la interfaz del plugin, como la adición de entradas opcionales, se realizarán de forma retrocompatible para no requerir cambios en los plugins existentes.

> **Note:** Todos los métodos con el prefijo `supports_` están reservados en una clase derivada de `UnitarySynthesisPlugin` como parte de la interfaz. No debes definir ningún método `supports_*` personalizado en una subclase que no esté definido en la clase abstracta.

Ahora, exponemos el plugin añadiendo un entry point en los metadatos de nuestro paquete de Python.
Aquí asumimos que la clase que definimos está expuesta en un módulo llamado `my_qiskit_plugin`, por ejemplo importándola en el archivo `__init__.py` del módulo `my_qiskit_plugin`.
Editamos el archivo `pyproject.toml`, `setup.cfg` o `setup.py` de nuestro paquete:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

Como antes, si tu proyecto usa `setup.cfg` o `setup.py` en lugar de `pyproject.toml`, consulta la [documentación de setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) para saber cómo adaptar estas líneas a tu situación.

Para verificar que Qiskit detecta correctamente tu plugin, instala el paquete del plugin y sigue las instrucciones en [Plugins del transpilador](transpiler-plugins#list-available-unitary-synthesis-plugins) para listar los plugins instalados, asegurándote de que tu plugin aparece en la lista:

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

Si nuestro plugin de ejemplo estuviera instalado, el nombre `my_unitary_synthesis` aparecería en esta lista.

Para dar cabida a plugins de síntesis unitaria que exponen múltiples opciones,
la interfaz del plugin permite a los usuarios proporcionar un diccionario de configuración libre.
Este se pasará al método `run` mediante el argumento de palabra clave `options`. Si tu plugin tiene estas opciones de configuración, debes documentarlas claramente.

## Ejemplo: Crear un plugin de síntesis de alto nivel

En este ejemplo, crearemos un plugin de síntesis de alto nivel que simplemente usa la función integrada [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) para sintetizar un operador Clifford.

La clase [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) define la interfaz y el contrato para los plugins de síntesis de alto nivel. El método principal es [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
El argumento posicional `high_level_object` es una [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) que representa el objeto de "alto nivel" a sintetizar. Por ejemplo, puede ser una
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) o un
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
Los siguientes argumentos de palabra clave están disponibles:
- `target` especifica el backend de destino, lo que permite al plugin
acceder a toda la información específica del destino,
como el mapa de acoplamiento, el conjunto de puertas admitidas, etc.
- `coupling_map` solo especifica el mapa de acoplamiento y se usa únicamente cuando no se especifica `target`.
- `qubits` especifica la lista de qubits sobre los que está definido el objeto de alto nivel, en caso de que la síntesis se realice sobre el circuito físico.
Un valor de ``None`` indica que el layout aún no ha sido elegido y que los qubits físicos del destino o del mapa de acoplamiento sobre los que opera esta operación todavía no han sido determinados.
- `options`, un diccionario de configuración libre para opciones específicas del plugin. Si tu plugin tiene estas opciones de configuración, debes documentarlas claramente.

El método `run` devuelve un [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
que representa el circuito sintetizado a partir del objeto de alto nivel.
También está permitido devolver `None`, lo que indica que el plugin no puede sintetizar el objeto de alto nivel dado.
La síntesis real de los objetos de alto nivel la realiza el paso del transpilador
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis).

Además del método `run`, hay una serie de métodos de propiedad que deben definirse.
Consulta [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) para ver la documentación de todas las propiedades requeridas.

Vamos a definir nuestra subclase de HighLevelSynthesisPlugin: